# NCAA Bracket Model — Modeling

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss



In [2]:
pd.set_option('display.max_rows', 100)

In [3]:
X_train = pd.read_parquet('../data/processed/X_train.parquet', engine='fastparquet')
X_val = pd.read_parquet('../data/processed/X_val.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet', engine='fastparquet').squeeze()
y_val = pd.read_parquet('../data/processed/y_val.parquet', engine='fastparquet').squeeze()
y_test = pd.read_parquet('../data/processed/y_test.parquet', engine='fastparquet').squeeze()

In [4]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(1248, 56) (1248,)
(67, 56) (67,)
(67, 56) (67,)


## Creating Logistic Regression Baseline Model

In [5]:
scalar = StandardScaler()
scalar.fit(X_train)
X_train_scaled = scalar.transform(X_train)
X_val_scaled = scalar.transform(X_val)
X_test_scaled = scalar.transform(X_test)

In [6]:
model = LogisticRegression(max_iter=1000, random_state = 0)
model.fit(X_train_scaled, y_train)
linear_predictions = model.predict(X_val_scaled)



In [7]:
# Metrics
linear_accuracy = accuracy_score(y_val, linear_predictions)
linear_logloss = log_loss(y_val, model.predict_proba(X_val_scaled))
linear_brier_score = brier_score_loss(y_val,model.predict_proba(X_val_scaled)[:,1])

linear_accuracy, linear_logloss, linear_brier_score

(0.5970149253731343, 0.6563119844426238, 0.22070852194993965)

## XGBoost Testing

#### v1 (base values)

In [8]:
model_v1 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=4, n_jobs=4, random_state=0)
model_v1.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v1.predict(X_val)


In [9]:
XGB1_accuracy = accuracy_score(y_val, XGB_predictions)
XGB1_logloss = log_loss(y_val, model_v1.predict_proba(X_val))
XGB1_brier_score = brier_score_loss(y_val, model_v1.predict_proba(X_val)[:,1])

XGB1_accuracy, XGB1_logloss, XGB1_brier_score

(0.7164179104477612, 0.5948187804154296, 0.20739375054836273)

#### v2 (testing max_depth)

In [10]:
model_v2 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=3, n_jobs=4, random_state=0)
model_v2.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v2.predict(X_val)


In [11]:
XGB2_accuracy = accuracy_score(y_val, XGB_predictions)
XGB2_logloss = log_loss(y_val, model_v2.predict_proba(X_val))
XGB2_brier_score = brier_score_loss(y_val, model_v2.predict_proba(X_val)[:,1])

XGB2_accuracy, XGB2_logloss, XGB2_brier_score

(0.6417910447761194, 0.6108600782631547, 0.2133203148841858)

#### v3 (testing learning_rate)

In [12]:
model_v3 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=3, n_jobs=4, random_state=0)
model_v3.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v3.predict(X_val)

In [13]:
XGB3_accuracy = accuracy_score(y_val, XGB_predictions)
XGB3_logloss = log_loss(y_val, model_v3.predict_proba(X_val))
XGB3_brier_score = brier_score_loss(y_val, model_v3.predict_proba(X_val)[:,1])

XGB3_accuracy, XGB3_logloss, XGB3_brier_score

(0.6417910447761194, 0.6108600782631547, 0.2133203148841858)

#### v4 (testing n_estimators)

In [14]:
model_v4 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=3, subsample=0.8, n_jobs=4, random_state=0)
model_v4.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v4.predict(X_val)

In [15]:
XGB4_accuracy = accuracy_score(y_val, XGB_predictions)
XGB4_logloss = log_loss(y_val, model_v4.predict_proba(X_val))
XGB4_brier_score = brier_score_loss(y_val, model_v4.predict_proba(X_val)[:,1])

XGB4_accuracy, XGB4_logloss, XGB4_brier_score

(0.6567164179104478, 0.6200749357660815, 0.21610400080680847)

#### v5 (adding/testing subsample)

In [16]:
model_v5 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=3, subsample = 0.7, n_jobs=4, random_state=0)
model_v5.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v5.predict(X_val)

In [17]:
XGB5_accuracy = accuracy_score(y_val, XGB_predictions)
XGB5_logloss = log_loss(y_val, model_v5.predict_proba(X_val))
XGB5_brier_score = brier_score_loss(y_val, model_v5.predict_proba(X_val)[:,1])

XGB5_accuracy, XGB5_logloss, XGB5_brier_score

(0.6417910447761194, 0.6065935754933784, 0.21060141921043396)

### FINAL MODEL

In [18]:
# model_v4 selected as final model (max_depth=3, subsample=0.8).
# Validation (2023): accuracy=65.67%, log_loss=0.620, brier=0.216.
# Test (2024):       accuracy=65.67%, log_loss=0.597, brier=0.205.
# Note: v1 (max_depth=4) had the highest validation accuracy at 71.64%;
# v5 (subsample=0.7) had the best validation log_loss and brier score.
# Revisit model selection if you want to optimise for a single metric.
final_model = model_v4
final_predictions = final_model.predict(X_test)
final_predictions

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0,
       1])

In [19]:
final_accuracy = accuracy_score(y_test, final_predictions)
final_logloss = log_loss(y_test, final_model.predict_proba(X_test))
final_brier_score = brier_score_loss(y_test, final_model.predict_proba(X_test)[:,1])

final_accuracy, final_logloss, final_brier_score

(0.6567164179104478, 0.5969736396979934, 0.2054518163204193)